In [ ]:
# Cell 1 [全局模型规模开关 + Git / WandB 配置 + 挂载 Google Drive 与 GPU 体检]
# ==============================================================================
# 单变量全局开关（供 Cell 2 / Cell 3 / Cell 5 / Cell 6 自动读取）：
#   "210m"   -> DCPA-Turbo 全并行皮层三明治 210M（v2 6域全科语料，32 experts Top-2 + 共享48KB环，零 Rust）
#   "48m_v2" -> DCPA-Turbo 48M v2 全科版（复用同一份 v2 6域语料，8 experts Top-2，checkpoint 独立隔离）
#   "48m"    -> DCPA-Turbo 48M v1 基线版（v1 5域语料，checkpoint 独立隔离）
# 代码同步：默认直接从 GitHub `git clone / git pull`（私有仓可在 Colab 左侧 🔑 Secrets 配 `GITHUB_TOKEN`）
# 实验追踪：默认启用 `wandb`（可在 Colab Secrets 配 `WANDB_API_KEY`；未配则自动降级 offline 不阻塞训练）
# 进度保存：Google Drive 持续保存按 MODEL_PRESET 隔离的 A/B 双槽断点、日志、导出权重，以及按语料体系+scale 隔离的归档
# ==============================================================================
MODEL_PRESET = "210m"        # 默认推荐 "210m"；可切 "48m_v2" 或 "48m"
GPU_PROFILE = "auto"         # 默认 "auto" 自动适配当前显卡；也可手动指定 "l4" / "g4" / "g4_256k" / "a100" / "a100_40g" / "h100" / "t4"
CORPUS_SCALE_FACTOR = 12.0   # v2 语料规模倍率：12.0 ≈ 5.28GB，4.0 ≈ 1.76GB，2.0 ≈ 880MB（按倍率独立归档）
SMOKE_TEST_ONLY = False      # True = 2 分钟全链路冒烟验证
GIT_REPO_URL = "https://github.com/Eveyz/dcpa.git"
GIT_BRANCH = "feat/dcpa-turbo-parallel"  # 合并主干后可改为 "main"
USE_WANDB = True             # True = 开启 Weights & Biases 实时监控 Loss / BPC / ACh / Pondering
WANDB_PROJECT = "dcpa-turbo"
WANDB_RUN_NAME = ""          # 留空自动命名（如 dcpa-turbo-moe_210m）；断连重连自动恢复同一 run_id
HF_TOKEN = ""                # 可选：留空则自动读 Colab Secrets (🔑) 中的 HF_TOKEN；可提速 HF 下载并解锁 gated 语料 (CCI3-HQ / xLAM)

import os
import subprocess
from pathlib import Path

if MODEL_PRESET not in ("48m", "48m_v2", "210m"):
    raise ValueError(f"MODEL_PRESET must be '48m', '48m_v2', or '210m', got {MODEL_PRESET!r}")
os.environ["DCPA_MODEL_PRESET"] = MODEL_PRESET
os.environ["DCPA_GPU_PROFILE"] = GPU_PROFILE.strip() or "auto"
os.environ["DCPA_CORPUS_SCALE"] = str(CORPUS_SCALE_FACTOR)
os.environ["DCPA_SMOKE_ONLY"] = "1" if SMOKE_TEST_ONLY else "0"
os.environ["DCPA_GIT_REPO_URL"] = GIT_REPO_URL
os.environ["DCPA_GIT_BRANCH"] = GIT_BRANCH
os.environ["DCPA_USE_WANDB"] = "1" if USE_WANDB else "0"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
if WANDB_RUN_NAME:
    os.environ["WANDB_NAME"] = WANDB_RUN_NAME
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path(os.environ.get("DCPA_DRIVE_ROOT", "/content/drive/MyDrive/DCPA_Turbo"))
os.environ["DCPA_DRIVE_ROOT"] = str(DRIVE_ROOT)
for sub in (
    "code",
    "data_archive",
    f"checkpoints/{MODEL_PRESET}",
    f"logs/{MODEL_PRESET}",
    f"export/{MODEL_PRESET}",
):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print(
    f"[Drive] Ready: {DRIVE_ROOT} (preset={MODEL_PRESET} isolated under checkpoints/logs/export/{MODEL_PRESET}) | "
    f"CORPUS_SCALE={CORPUS_SCALE_FACTOR}x | GIT={GIT_REPO_URL}@{GIT_BRANCH or 'default'} | "
    f"USE_WANDB={USE_WANDB} | SMOKE_TEST_ONLY={SMOKE_TEST_ONLY}"
)

# GPU health check & 4-tier hardware sharding preview (no hardcoded torch.cuda.get_device_properties)
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True,
    text=True,
    check=False,
)
if smi.returncode == 0 and smi.stdout.strip():
    first_line = smi.stdout.strip().splitlines()[0]
    gpu_name, mem_mib_s = [part.strip() for part in first_line.split(",", 1)]
    vram_gb = float(mem_mib_s) / 1024.0
    if vram_gb >= 88.0:
        tier, micro_bytes, accum = "blackwell_or_h200_96g (g4)", 131072, 1
    elif vram_gb >= 75.0:
        tier, micro_bytes, accum = "h100_or_a100_80g", 131072, 1
    elif vram_gb >= 38.0:
        tier, micro_bytes, accum = "a100_40g", 65536, 2
    elif vram_gb >= 30.0:
        tier, micro_bytes, accum = "rtx_5090_32g", 65536, 2
    elif vram_gb >= 20.0:
        tier, micro_bytes, accum = "l4_or_4090 (l4)", 65536, 2
    else:
        tier, micro_bytes, accum = "cuda_compact (t4)", 16384, 8
    print(
        f"[GPU Health] {gpu_name} | VRAM={vram_gb:.1f} GB | profile={GPU_PROFILE} | tier={tier} | "
        f"micro_batch_bytes={micro_bytes} | grad_accum_steps={accum} | precision=bfloat16"
    )
else:
    print("[GPU Health][WARN] No NVIDIA GPU detected via nvidia-smi; will fall back to CPU float32.")


In [ ]:
# Cell 2 [Git 一键拉取 / 更新代码至本地 NVMe（告别手动传 zip！）与安装依赖]
import base64
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

CODE_DIR = Path("/content/dcpa")
NVME_CACHE = Path("/content/nvme_cache")
DRIVE_ROOT = Path(os.environ.get("DCPA_DRIVE_ROOT", "/content/drive/MyDrive/DCPA_Turbo"))
LEGACY_DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
BUNDLE_ZIP = DRIVE_ROOT / "code/dcpa_colab_bundle.zip"
if not BUNDLE_ZIP.exists() and (LEGACY_DRIVE_ROOT / "code/dcpa_colab_bundle.zip").exists():
    BUNDLE_ZIP = LEGACY_DRIVE_ROOT / "code/dcpa_colab_bundle.zip"
NVME_CACHE.mkdir(parents=True, exist_ok=True)

repo_url = os.environ.get("DCPA_GIT_REPO_URL", "https://github.com/Eveyz/dcpa.git").strip()
git_branch = os.environ.get("DCPA_GIT_BRANCH", "feat/dcpa-turbo-parallel").strip()
gh_token = os.environ.get("GITHUB_TOKEN", "").strip()
if not gh_token:
    try:
        from google.colab import userdata
        gh_token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        gh_token = ""
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        wb_key = (userdata.get("WANDB_API_KEY") or "").strip()
        if wb_key:
            os.environ["WANDB_API_KEY"] = wb_key
    except Exception:
        pass
if not os.environ.get("HF_TOKEN"):
    for _hf_k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        try:
            from google.colab import userdata
            hf_tok = (userdata.get(_hf_k) or "").strip()
            if hf_tok:
                os.environ["HF_TOKEN"] = hf_tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_tok
                break
        except Exception:
            continue

git_prefix = ["git"]
if gh_token and repo_url.startswith("https://"):
    b64 = base64.b64encode(f"x-access-token:{gh_token}".encode("utf-8")).decode("ascii")
    git_prefix = ["git", "-c", f"http.extraHeader=Authorization: Basic {b64}"]

synced_via = ""
if repo_url:
    try:
        if (CODE_DIR / ".git").exists():
            subprocess.run([*git_prefix, "-C", str(CODE_DIR), "remote", "set-url", "origin", repo_url], check=True)
            subprocess.run([*git_prefix, "-C", str(CODE_DIR), "fetch", "--prune", "origin"], check=True)
            if git_branch:
                rc = subprocess.run([*git_prefix, "-C", str(CODE_DIR), "checkout", git_branch], check=False)
                if rc.returncode != 0:
                    subprocess.run([*git_prefix, "-C", str(CODE_DIR), "checkout", "-B", git_branch, f"origin/{git_branch}"], check=True)
                subprocess.run([*git_prefix, "-C", str(CODE_DIR), "pull", "--ff-only", "origin", git_branch], check=True)
            else:
                subprocess.run([*git_prefix, "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
            synced_via = "git pull --ff-only"
        else:
            if CODE_DIR.exists():
                shutil.rmtree(CODE_DIR)
            clone_cmd = [*git_prefix, "clone"]
            if git_branch:
                clone_cmd.extend(["--branch", git_branch])
            clone_cmd.extend([repo_url, str(CODE_DIR)])
            subprocess.run(clone_cmd, check=True)
            synced_via = "git clone"
        head_sha = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
        synced_via = f"{synced_via} ({git_branch or 'HEAD'} @ {head_sha})"
    except Exception as git_err:
        if BUNDLE_ZIP.exists():
            print(f"[Git][WARN] Git sync failed ({git_err}); falling back to Drive bundle {BUNDLE_ZIP}")
        else:
            raise RuntimeError(
                f"Git clone/pull from {repo_url} (branch={git_branch!r}) failed! "
                "If the repo is private, add `GITHUB_TOKEN` to Colab Secrets (🔑)."
            ) from git_err

if not synced_via:
    if not BUNDLE_ZIP.exists():
        raise FileNotFoundError(f"Neither Git repo ({repo_url}) nor fallback bundle ({BUNDLE_ZIP}) available.")
    if CODE_DIR.exists():
        shutil.rmtree(CODE_DIR)
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
        zf.extractall(CODE_DIR)
    synced_via = f"Drive fallback ({BUNDLE_ZIP.name})"

os.chdir(CODE_DIR)
for p in (str(CODE_DIR), str(CODE_DIR / "dcpa-train-py")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{CODE_DIR}:{CODE_DIR / 'dcpa-train-py'}"

deps = [
    "safetensors",
    "datasets",
    "hf_transfer",
    "wandb",
    "tree-sitter",
    "tree-sitter-python",
    "tree-sitter-javascript",
    "tree-sitter-rust",
    "pyyaml",
    "pytest",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", str(CODE_DIR / "dcpa-train-py")],
    check=True,
)

from utils.device import describe_hardware_profile, detect_hardware_profile

profile = detect_hardware_profile("auto")
print(
    f"[Deploy] Synced {CODE_DIR} via {synced_via} | "
    f"WANDB_KEY={'SET' if os.environ.get('WANDB_API_KEY') else 'offline/unset'} | "
    f"HF_TOKEN={'SET' if os.environ.get('HF_TOKEN') else 'unset'}"
)
print(describe_hardware_profile(profile))


In [ ]:
# Cell 3 [全科认知 / 基线底座与能力语料智能缓存：首次流式构建归档，重连极速恢复]
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

from train.colab_drive_sync import sync_file_to_drive
from data.hf_auth import configure_hf_auth

configure_hf_auth()

MODEL_PRESET = os.environ.get("DCPA_MODEL_PRESET", "210m")
CORPUS_SCALE = os.environ.get("DCPA_CORPUS_SCALE", "12.0")
SMOKE_ONLY = os.environ.get("DCPA_SMOKE_ONLY", "0") == "1"
IS_V2_COG = MODEL_PRESET in ("210m", "48m_v2")
_sc_f = float(CORPUS_SCALE)
_sc_tag = f"{int(_sc_f)}x" if _sc_f.is_integer() else f"{_sc_f:g}x"

DRIVE_ROOT = Path(os.environ.get("DCPA_DRIVE_ROOT", "/content/drive/MyDrive/DCPA_Turbo"))
ARCHIVE_DIR = DRIVE_ROOT / "data_archive"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
LEGACY_ARCHIVE_DIR = Path("/content/drive/MyDrive/DCPA_Project/data_archive")
if IS_V2_COG:
    suf = "_smoke" if SMOKE_ONLY else ""
    arch_name = "dcpa_corpus_v2_cognitive_smoke.tar.gz" if SMOKE_ONLY else f"dcpa_corpus_v2_cognitive_{_sc_tag}.tar.gz"
    legacy_12x_name = "dcpa_corpus_v2_cognitive.tar.gz" if (not SMOKE_ONLY and abs(_sc_f - 12.0) < 1e-6) else None
    cap_name = f"dcpa_cap_v2_cognitive{suf}.tar.gz"
    CORPUS_KEY = f"v2_cognitive_smoke" if SMOKE_ONLY else f"v2_cognitive_{_sc_tag}"
else:
    arch_name = "dcpa_corpus_v1.tar.gz"
    legacy_12x_name = None
    cap_name = "dcpa_cap_v1.tar.gz"
    CORPUS_KEY = "v1"

def _resolve_archive(primary_name: str, compat_name: str | None = None) -> Path:
    cands = [ARCHIVE_DIR / primary_name, LEGACY_ARCHIVE_DIR / primary_name]
    if compat_name:
        cands.extend([ARCHIVE_DIR / compat_name, LEGACY_ARCHIVE_DIR / compat_name])
    for c in cands:
        if c.exists():
            return c
    return ARCHIVE_DIR / primary_name

DRIVE_ARCHIVE = _resolve_archive(arch_name, legacy_12x_name)
DRIVE_CAP_ARCHIVE = _resolve_archive(cap_name)

NVME_DATA_DIR = Path(f"/content/nvme_cache/data_{CORPUS_KEY}")
NVME_CAP_DIR = Path(f"/content/nvme_cache/cap_{'v2_cognitive' if IS_V2_COG else 'v1'}{'_smoke' if (IS_V2_COG and SMOKE_ONLY) else ''}")
NVME_TAR_TMP = Path(f"/content/nvme_cache/{DRIVE_ARCHIVE.name}")
NVME_CAP_TAR_TMP = Path(f"/content/nvme_cache/{DRIVE_CAP_ARCHIVE.name}")

NVME_DATA_DIR.mkdir(parents=True, exist_ok=True)
NVME_CAP_DIR.mkdir(parents=True, exist_ok=True)

def _stream_run(cmd, env=None, check=True):
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = int(proc.wait())
    if check and rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)
    return rc

t0 = time.perf_counter()
smoke_flag = ["--smoke-quota"] if SMOKE_ONLY else []

# 1. Base pretraining corpus
if DRIVE_ARCHIVE.exists():
    print(f"[Corpus Cache HIT] Restoring {DRIVE_ARCHIVE} -> {NVME_DATA_DIR} ...")
    shutil.copy2(DRIVE_ARCHIVE, NVME_TAR_TMP)
    with tarfile.open(NVME_TAR_TMP, "r:gz") as tf:
        tf.extractall(NVME_DATA_DIR)
    NVME_TAR_TMP.unlink(missing_ok=True)
    print(f"[Corpus Cache HIT] Restored in {time.perf_counter() - t0:.2f}s.")
else:
    if IS_V2_COG:
        print(f"[Corpus Cache MISS] Building 6-domain v2 holistic corpus (scale={CORPUS_SCALE}x, zero Rust) on NVMe...")
        _stream_run([
            sys.executable, "-u", "data/build_corpus.py",
            "--out-dir", str(NVME_DATA_DIR),
            "--cognitive-v2", "--workers", "4", "--scale-factor", CORPUS_SCALE,
            *smoke_flag,
        ])
    else:
        print("[Corpus Cache MISS] Building ~450 MB v1 base corpus directly on local NVMe...")
        _stream_run([sys.executable, "-u", "data/build_corpus.py", "--out-dir", str(NVME_DATA_DIR), *smoke_flag])
    env = dict(os.environ)
    env["DCPA_DATA_DIR"] = str(NVME_DATA_DIR)
    _stream_run([sys.executable, "-u", "-m", "pytest", "-v", "tests/test_data_pipeline.py"], env=env)
    with tarfile.open(NVME_TAR_TMP, "w:gz") as tf:
        for fname in ("train.bin", "train.idx", "train.meta.json", "val.bin", "val.idx", "val.meta.json", "corpus_meta.json"):
            fpath = NVME_DATA_DIR / fname
            if fpath.exists():
                tf.add(fpath, arcname=fname)
    sync_file_to_drive(NVME_TAR_TMP, DRIVE_ARCHIVE.parent, dest_name=DRIVE_ARCHIVE.name)
    NVME_TAR_TMP.unlink(missing_ok=True)
    print(f"[Corpus] Archived to {DRIVE_ARCHIVE} in {time.perf_counter() - t0:.1f}s.")

# 2. Capability & Data-Alchemy Shards
t1 = time.perf_counter()
if DRIVE_CAP_ARCHIVE.exists():
    print(f"[Cap Cache HIT] Restoring {DRIVE_CAP_ARCHIVE} -> {NVME_CAP_DIR} ...")
    shutil.copy2(DRIVE_CAP_ARCHIVE, NVME_CAP_TAR_TMP)
    with tarfile.open(NVME_CAP_TAR_TMP, "r:gz") as tf:
        tf.extractall(NVME_CAP_DIR)
    NVME_CAP_TAR_TMP.unlink(missing_ok=True)
    print(f"[Cap Cache HIT] Restored capability shards in {time.perf_counter() - t1:.2f}s.")
else:
    if IS_V2_COG:
        print("[Cap Cache MISS] Building 520 MB v2 Alchemy shards (reasoning 160 + writing 160 + code 120 + instruct 40 + toolcall 40)...")
        _stream_run([sys.executable, "-u", "data/build_cap_reasoning.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "160", "--cognitive-v2", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_writing.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "160", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_code_v2.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "120", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_instruct.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "40", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_toolcall.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "40", *smoke_flag])
    else:
        print("[Cap Cache MISS] Streaming 260 MB v1 capability shards (reasoning 100MB + writing 100MB + toolcall 60MB)...")
        _stream_run([sys.executable, "-u", "data/build_cap_reasoning.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "100", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_writing.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "100", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_toolcall.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "60", *smoke_flag])
    with tarfile.open(NVME_CAP_TAR_TMP, "w:gz") as tf:
        for fpath in sorted(NVME_CAP_DIR.iterdir()):
            if fpath.is_file():
                tf.add(fpath, arcname=fpath.name)
    sync_file_to_drive(NVME_CAP_TAR_TMP, DRIVE_CAP_ARCHIVE.parent, dest_name=DRIVE_CAP_ARCHIVE.name)
    NVME_CAP_TAR_TMP.unlink(missing_ok=True)
    if hasattr(os, "sync"):
        os.sync()
    print(f"[Cap] Archived capability shards to {DRIVE_CAP_ARCHIVE} in {time.perf_counter() - t1:.1f}s.")


In [ ]:
# Cell 4 [云端 CUDA 算子与 DCPA v2 全科认知管线上线复检]
import subprocess
import sys
import time

t0 = time.perf_counter()
cmd = [
    sys.executable,
    "-u",
    "-m",
    "pytest",
    "-v",
    "tests/test_sanity.py",
    "tests/test_trainer_lifecycle.py",
    "tests/test_capabilities_upgrade.py",
    "tests/test_v2_cognitive_pipeline.py",
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
rc = int(proc.wait())
if rc != 0:
    raise RuntimeError(f"Cloud preflight pytest failed with code {rc}")
print(f"[Cloud Preflight] Causality, bf16 Islands, STE gradients, Capability Upgrade & v2 Cognitive 100% GREEN in {time.perf_counter() - t0:.1f}s.")


In [ ]:
# Cell 5 [一键启动 / 断连无缝续跑 10 万步底座训练（支持 v2 全科 210M / 基线 48M）]
# 注：自动释放显卡逻辑已移至 Cell 6（全部 Stage 1~5 任务完成后统一释放）
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

MODEL_PRESET = os.environ.get("DCPA_MODEL_PRESET", "210m")
CORPUS_SCALE = os.environ.get("DCPA_CORPUS_SCALE", "12.0")
SMOKE_ONLY = os.environ.get("DCPA_SMOKE_ONLY", "0") == "1"
IS_V2_COG = MODEL_PRESET in ("210m", "48m_v2")
_sc_f = float(CORPUS_SCALE)
_sc_tag = f"{int(_sc_f)}x" if _sc_f.is_integer() else f"{_sc_f:g}x"
if MODEL_PRESET == "210m":
    CFG_FILE, PRESET_NAME = "configs/v2/dcpa_210m_v2_cognitive.yaml", "moe_210m"
elif MODEL_PRESET == "48m_v2":
    CFG_FILE, PRESET_NAME = "configs/v2/dcpa_48m_v2_cognitive.yaml", "proto_48m"
else:
    CFG_FILE, PRESET_NAME = "configs/v1/dcpa_48m_train.yaml", "proto_48m"

if IS_V2_COG:
    suf = "_smoke" if SMOKE_ONLY else ""
    DRIVE_ARCHIVE_NAME = "dcpa_corpus_v2_cognitive_smoke.tar.gz" if SMOKE_ONLY else f"dcpa_corpus_v2_cognitive_{_sc_tag}.tar.gz"
    LEGACY_12X_NAME = "dcpa_corpus_v2_cognitive.tar.gz" if (not SMOKE_ONLY and abs(_sc_f - 12.0) < 1e-6) else None
    DRIVE_CAP_NAME = f"dcpa_cap_v2_cognitive{suf}.tar.gz"
    CAP_SENTINEL = "cap_code_train.bin"
    CORPUS_KEY = "v2_cognitive_smoke" if SMOKE_ONLY else f"v2_cognitive_{_sc_tag}"
else:
    DRIVE_ARCHIVE_NAME = "dcpa_corpus_v1.tar.gz"
    LEGACY_12X_NAME = None
    DRIVE_CAP_NAME = "dcpa_cap_v1.tar.gz"
    CAP_SENTINEL = "cap_reasoning_train.bin"
    CORPUS_KEY = "v1"

DRIVE_ROOT = Path(os.environ.get("DCPA_DRIVE_ROOT", "/content/drive/MyDrive/DCPA_Turbo"))
os.environ["DCPA_DRIVE_ROOT"] = str(DRIVE_ROOT)
ARCHIVE_DIR = DRIVE_ROOT / "data_archive"
LEGACY_ARCHIVE_DIR = Path("/content/drive/MyDrive/DCPA_Project/data_archive")
NVME_CACHE = Path("/content/nvme_cache")
NVME_DATA_DIR = NVME_CACHE / f"data_{CORPUS_KEY}"
NVME_CAP_DIR = NVME_CACHE / f"cap_{'v2_cognitive' if IS_V2_COG else 'v1'}{'_smoke' if (IS_V2_COG and SMOKE_ONLY) else ''}"

def _find_archive(primary_name: str, compat_name: str | None = None) -> Path:
    cands = [ARCHIVE_DIR / primary_name, LEGACY_ARCHIVE_DIR / primary_name]
    if compat_name:
        cands.extend([ARCHIVE_DIR / compat_name, LEGACY_ARCHIVE_DIR / compat_name])
    for c in cands:
        if c.exists():
            return c
    return ARCHIVE_DIR / primary_name

DRIVE_ARCHIVE = _find_archive(DRIVE_ARCHIVE_NAME, LEGACY_12X_NAME)
DRIVE_CAP_ARCHIVE = _find_archive(DRIVE_CAP_NAME)
# 按 MODEL_PRESET 独立隔离 checkpoints / logs / export，切换模型规格互不覆盖！
DRIVE_CKPTS = DRIVE_ROOT / "checkpoints" / MODEL_PRESET
DRIVE_LOGS = DRIVE_ROOT / "logs" / MODEL_PRESET
DRIVE_EXPORT = DRIVE_ROOT / "export" / MODEL_PRESET
for _d in (DRIVE_CKPTS, DRIVE_LOGS, DRIVE_EXPORT):
    _d.mkdir(parents=True, exist_ok=True)
FINAL_SAFETENSORS = DRIVE_EXPORT / "dcpa_final.safetensors"

# 断连重连防呆保护：若新实例 NVMe 尚未解压语料而 Drive 已有归档，自动恢复至本地 NVMe！
NVME_DATA_DIR.mkdir(parents=True, exist_ok=True)
NVME_CAP_DIR.mkdir(parents=True, exist_ok=True)
if not (NVME_DATA_DIR / "train.bin").exists() and DRIVE_ARCHIVE.exists():
    print(f"[Auto-Restore] NVMe train.bin missing; restoring {DRIVE_ARCHIVE} -> {NVME_DATA_DIR} ...", flush=True)
    tmp_tar = NVME_CACHE / "_auto_corpus.tar.gz"
    shutil.copy2(DRIVE_ARCHIVE, tmp_tar)
    with tarfile.open(tmp_tar, "r:gz") as tf:
        tf.extractall(NVME_DATA_DIR)
    tmp_tar.unlink(missing_ok=True)
if not (NVME_CAP_DIR / CAP_SENTINEL).exists() and DRIVE_CAP_ARCHIVE.exists():
    print(f"[Auto-Restore] NVMe cap_data missing; restoring {DRIVE_CAP_ARCHIVE} -> {NVME_CAP_DIR} ...", flush=True)
    tmp_cap = NVME_CACHE / "_auto_cap.tar.gz"
    shutil.copy2(DRIVE_CAP_ARCHIVE, tmp_cap)
    with tarfile.open(tmp_cap, "r:gz") as tf:
        tf.extractall(NVME_CAP_DIR)
    tmp_cap.unlink(missing_ok=True)

total_steps = 200 if SMOKE_ONLY else (60000 if MODEL_PRESET == "48m_v2" else 100000)
stage_args = ["--stage-boundaries", "0,30,90,150,200"] if SMOKE_ONLY else []
seq_args = [] if IS_V2_COG else ["--seq-len", "2048"]
save_every = 50 if SMOKE_ONLY else 1000
probe_every = 50 if SMOKE_ONLY else 500

cmd = [
    sys.executable,
    "-u",
    "train/train_dcpa.py",
    "--gpus", "1",
    "--gpu-profile", os.environ.get("DCPA_GPU_PROFILE", "auto"),
    "--config", CFG_FILE,
    "--preset", PRESET_NAME,
    "--total-steps", str(total_steps),
    *stage_args,
    *seq_args,
    "--data-dir", str(NVME_DATA_DIR),
    "--cap-data-dir", str(NVME_CAP_DIR),
    "--local-cache-dir", str(NVME_CACHE),
    "--drive-sync-dir", str(DRIVE_CKPTS),
    "--drive-logs-dir", str(DRIVE_LOGS),
    "--drive-export-dir", str(DRIVE_EXPORT),
    "--save-every-steps", str(save_every),
    "--save-every-minutes", "10",
    "--probe-every-steps", str(probe_every),
    "--heartbeat-seconds", "60",
    "--resume", "auto",
    "--compile-mode", "auto",
    "--max-consecutive-meltdowns", "3",
]
if os.environ.get("DCPA_USE_WANDB", "1") == "1":
    cmd.extend(["--wandb", "--wandb-project", os.environ.get("WANDB_PROJECT", "dcpa-turbo")])
    if os.environ.get("WANDB_NAME"):
        cmd.extend(["--wandb-run-name", os.environ["WANDB_NAME"]])

print(f"[Train][{MODEL_PRESET}] Launching: {' '.join(cmd)}", flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
exit_code = int(proc.wait())
if hasattr(os, "sync"):
    os.sync()
if exit_code == 0:
    print(f"[Train] {total_steps:,} steps ({PRESET_NAME}) completed cleanly! Final export: {FINAL_SAFETENSORS}", flush=True)
elif exit_code == 42:
    raise RuntimeError("[Train][CIRCUIT_BREAKER] Aborted after 3 consecutive unrecoverable meltdowns.")
else:
    raise RuntimeError(f"Training exited unexpectedly with code {exit_code}")


In [ ]:
# Cell 6 [Stage 5：15,000 步全科多稿打磨 / 苏格拉底思辨 / 排错微调与导出 + 完工自动释放显卡]
# ==============================================================================
# AUTO_UNASSIGN_RUNTIME_ON_FINISH:
#   True  -> Stage 5 跑满后导出 dcpa_chat_sft.safetensors 至 Google Drive，
#            强制 os.sync() 等待 10 秒并自动归还 H100/A100 实例，防止空烧 CCU。
#   False -> 保留当前 Colab 实例以便手动调试。
# ==============================================================================
AUTO_UNASSIGN_RUNTIME_ON_FINISH = True

import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

from train.colab_drive_sync import sync_file_to_drive
from data.hf_auth import configure_hf_auth

configure_hf_auth()

MODEL_PRESET = os.environ.get("DCPA_MODEL_PRESET", "210m")
SMOKE_ONLY = os.environ.get("DCPA_SMOKE_ONLY", "0") == "1"
IS_V2_COG = MODEL_PRESET in ("210m", "48m_v2")
DRIVE_ROOT = Path(os.environ.get("DCPA_DRIVE_ROOT", "/content/drive/MyDrive/DCPA_Turbo"))
os.environ["DCPA_DRIVE_ROOT"] = str(DRIVE_ROOT)
LEGACY_ARCHIVE_DIR = Path("/content/drive/MyDrive/DCPA_Project/data_archive")
NVME_CACHE = Path("/content/nvme_cache")
NVME_SFT_DIR = NVME_CACHE / f"sft_{'v2_cognitive' if IS_V2_COG else 'v1'}{'_smoke' if (IS_V2_COG and SMOKE_ONLY) else ''}"
NVME_CAP_DIR = NVME_CACHE / f"cap_{'v2_cognitive' if IS_V2_COG else 'v1'}{'_smoke' if (IS_V2_COG and SMOKE_ONLY) else ''}"
if IS_V2_COG:
    suf = "_smoke" if SMOKE_ONLY else ""
    sft_arch_name = f"dcpa_sft_stage5_v2_cognitive{suf}.tar.gz"
    SFT_STEPS = "50" if SMOKE_ONLY else ("15000" if MODEL_PRESET == "210m" else "10000")
    MAX_DOC_BYTES = "4096"
else:
    sft_arch_name = "dcpa_sft_stage5_v1.tar.gz"
    SFT_STEPS = "50" if SMOKE_ONLY else "8000"
    MAX_DOC_BYTES = "2048"
DRIVE_SFT_ARCHIVE = DRIVE_ROOT / "data_archive" / sft_arch_name
if not DRIVE_SFT_ARCHIVE.exists() and (LEGACY_ARCHIVE_DIR / sft_arch_name).exists():
    DRIVE_SFT_ARCHIVE = LEGACY_ARCHIVE_DIR / sft_arch_name
DRIVE_EXPORT = DRIVE_ROOT / "export" / MODEL_PRESET
DRIVE_EXPORT.mkdir(parents=True, exist_ok=True)
BASE_WEIGHTS = DRIVE_EXPORT / "dcpa_final.safetensors"
BASE_CONFIG = DRIVE_EXPORT / "config.json"
CHAT_WEIGHTS = DRIVE_EXPORT / "dcpa_chat_sft.safetensors"

exit_code = -1
try:
    if not BASE_WEIGHTS.exists():
        raise FileNotFoundError(f"Missing Stage 4 base export: {BASE_WEIGHTS}. Run Cell 5 first.")

    # 1. Build or restore Stage 5 SFT dataset on local NVMe
    NVME_SFT_DIR.mkdir(parents=True, exist_ok=True)
    if DRIVE_SFT_ARCHIVE.exists():
        tmp_tar = NVME_CACHE / "sft_stage5.tar.gz"
        shutil.copy2(DRIVE_SFT_ARCHIVE, tmp_tar)
        with tarfile.open(tmp_tar, "r:gz") as tf:
            tf.extractall(NVME_SFT_DIR)
        tmp_tar.unlink(missing_ok=True)
        print(f"[Stage 5 Data] Restored cached SFT corpus -> {NVME_SFT_DIR}", flush=True)
    else:
        build_sft_cmd = [sys.executable, "-u", "data/build_sft_stage5.py", "--out-dir", str(NVME_SFT_DIR)]
        if IS_V2_COG:
            build_sft_cmd.extend(["--cognitive-v2", "--train-mb", "80", "--val-mb", "4"])
        if SMOKE_ONLY:
            build_sft_cmd.append("--smoke-quota")
        b_proc = subprocess.Popen(
            build_sft_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert b_proc.stdout is not None
        for line in b_proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
        if int(b_proc.wait()) != 0:
            raise RuntimeError("Stage 5 SFT dataset build failed!")
        tmp_tar = NVME_CACHE / "sft_stage5.tar.gz"
        with tarfile.open(tmp_tar, "w:gz") as tf:
            for fpath in sorted(NVME_SFT_DIR.iterdir()):
                if fpath.is_file():
                    tf.add(fpath, arcname=fpath.name)
        sync_file_to_drive(tmp_tar, DRIVE_SFT_ARCHIVE.parent, dest_name=DRIVE_SFT_ARCHIVE.name)
        tmp_tar.unlink(missing_ok=True)

    # 2. Run Stage 5 SFT (15,000 steps for v2 cognitive 210M / 8,000 steps for v1 48M)
    sft_cmd = [
        sys.executable,
        "-u",
        "train/train_stage5_sft.py",
        "--gpus", "1",
        "--weights", str(BASE_WEIGHTS),
        "--data-dir", str(NVME_SFT_DIR),
        "--cap-data-dir", str(NVME_CAP_DIR),
        "--export-dir", str(NVME_CACHE / "export"),
        "--drive-export-dir", str(DRIVE_EXPORT),
        "--total-steps", SFT_STEPS,
        "--target-tokens-bytes-per-step", "65536",
        "--max-doc-bytes", MAX_DOC_BYTES,
        "--compile-mode", "auto",
    ]
    if BASE_CONFIG.exists():
        sft_cmd.extend(["--config", str(BASE_CONFIG)])
    if os.environ.get("DCPA_USE_WANDB", "1") == "1":
        sft_cmd.extend(["--wandb", "--wandb-project", os.environ.get("WANDB_PROJECT", "dcpa-turbo")])
        if os.environ.get("WANDB_NAME"):
            sft_cmd.extend(["--wandb-run-name", f"{os.environ['WANDB_NAME']}-sft"])
    print(f"[Stage 5 SFT] Launching: {' '.join(sft_cmd)}", flush=True)
    proc = subprocess.Popen(sft_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    exit_code = int(proc.wait())
    if exit_code == 0 and CHAT_WEIGHTS.exists():
        print(f"[Stage 5 DONE] Chat + Tool SFT weights synced to Drive: {CHAT_WEIGHTS}", flush=True)
    else:
        raise RuntimeError(f"Stage 5 SFT exited with code {exit_code}")
finally:
    if hasattr(os, "sync"):
        os.sync()
    if AUTO_UNASSIGN_RUNTIME_ON_FINISH and exit_code == 0 and CHAT_WEIGHTS.exists():
        print("[Auto-Unassign] Waiting 10s after os.sync() before releasing GPU...", flush=True)
        time.sleep(10)
        from google.colab import runtime

        runtime.unassign()
